In [1]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from keras.layers import Dense, Input, Conv2D, MaxPooling2D, Flatten,Dropout
from keras.utils import to_categorical
from keras import models
from keras.models import Model
import tensorflow as tf
from keras import layers
from keras.callbacks import EarlyStopping

In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "PetImages",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(150,150),
    batch_size=32,
    color_mode="rgb"
)

Found 24998 files belonging to 2 classes.
Using 19999 files for training.


In [3]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    "PetImages",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(150,150),
    batch_size=32,
    color_mode="rgb"
)

Found 24998 files belonging to 2 classes.
Using 4999 files for validation.


In [4]:
#Normalization layer

normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(
    lambda x,y : (normalization_layer(x),y)
)

val_ds = val_ds.map(
    lambda x,y : (normalization_layer(x),y)
)

In [5]:
model = models.Sequential()
model.add(Input(shape=(150,150,3)))
model.add(Conv2D(32,(3,3),activation = "relu",padding = "valid"))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64,(3,3),activation = "relu",padding = "valid"))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128,(3,3),activation = "relu",padding = "valid"))
model.add(MaxPooling2D(2,2))

model.add(Flatten())
model.add(Dense(128,activation = "relu"))
model.add(Dropout(0.25))
model.add(Dense(1,activation = "sigmoid"))

In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,735,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,828,481 (18.42 MB)

 Trainable params: 4,828,481 (18.42 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
earlystop = EarlyStopping(
    monitor = "accuracy",
    patience = 3,
    mode = "max",
    restore_best_weights = True
)

In [8]:
model.compile(
    loss = "binary_crossentropy",
    optimizer = "adam",
    metrics = ["accuracy"]    
)

In [9]:
model.fit(
    train_ds,
    batch_size = 32,
    epochs = 6,
    callbacks = earlystop,
    verbose = 1,
    validation_data = (val_ds)
)

Epoch 1/6
625/625 ━━━━━━━━━━━━━━━━━━━━ 336s 535ms/step - accuracy: 0.6556 - loss: 0.6116 - val_accuracy: 0.7231 - val_loss: 0.5392
Epoch 2/6
625/625 ━━━━━━━━━━━━━━━━━━━━ 391s 625ms/step - accuracy: 0.7723 - loss: 0.4717 - val_accuracy: 0.7992 - val_loss: 0.4386
Epoch 3/6
625/625 ━━━━━━━━━━━━━━━━━━━━ 360s 576ms/step - accuracy: 0.8292 - loss: 0.3838 - val_accuracy: 0.8242 - val_loss: 0.3914
Epoch 4/6
625/625 ━━━━━━━━━━━━━━━━━━━━ 404s 647ms/step - accuracy: 0.8593 - loss: 0.3187 - val_accuracy: 0.8190 - val_loss: 0.4268
Epoch 5/6
625/625 ━━━━━━━━━━━━━━━━━━━━ 429s 686ms/step - accuracy: 0.8963 - loss: 0.2466 - val_accuracy: 0.8412 - val_loss: 0.4125
Epoch 6/6
625/625 ━━━━━━━━━━━━━━━━━━━━ 318s 509ms/step - accuracy: 0.9258 - loss: 0.1769 - val_accuracy: 0.8440 - val_loss: 0.4655


In [10]:
model.save("cats_dogs_classifier.keras")

In [31]:
#load image
img = tf.keras.utils.load_img(
    "cat_jpg.jpg",
    target_size = (150,150)

)
#Convert it into array
img_array = tf.keras.utils.img_to_array(img)

#Convert it into the CNN model dimension
img_array = np.expand_dims(img_array,axis=0)

In [32]:
print(img_array.shape)

(1, 150, 150, 3)


In [42]:
prediction1 = model.predict(img_array)
if prediction1[0][0] >0.5:
    print("the image is given is Dog")
else:
    print("the image is given is Cat")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
the image is given is Cat


In [43]:
img2 = tf.keras.utils.load_img(
    "dog.jpg",
    target_size = (150,150)

)
#Convert it into array
img2_array = tf.keras.utils.img_to_array(img2)

#Convert it into the CNN model dimension
img2_array = np.expand_dims(img2_array,axis=0)

In [46]:
prediction2 = model.predict(img2_array)
if prediction2[0][0] >0.5:
    print("the image is given is Dog")
else:
    print("the image is given is Cat")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
the image is given is Dog
